In [0]:
USE CATALOG tibia_analytics;
USE SCHEMA bronze;
SET TIME ZONE 'UTC';

In [0]:
COPY INTO bronze.raw_worlds
FROM (
  SELECT
    sha1(concat_ws('|', _metadata.file_path, cast(_metadata.file_modification_time AS STRING))) AS raw_worlds_id, 
    from_json(
      to_json(worlds),
      'STRUCT<
        players_online: INT,
        record_players: INT,
        record_date: STRING,
        regular_worlds: ARRAY<STRUCT<
          name: STRING,
          status: STRING,
          players_online: INT,
          location: STRING,
          pvp_type: STRING,
          premium_only: BOOLEAN,
          transfer_type: STRING,
          battleye_protected: BOOLEAN,
          battleye_date: STRING,
          game_world_type: STRING,
          tournament_world_type: STRING
        >>,
        tournament_worlds: ARRAY<STRUCT<
          name: STRING,
          status: STRING,
          players_online: INT,
          location: STRING,
          pvp_type: STRING,
          premium_only: BOOLEAN,
          transfer_type: STRING,
          battleye_protected: BOOLEAN,
          battleye_date: STRING,
          game_world_type: STRING,
          tournament_world_type: STRING
        >>
      >'
    ) AS worlds,
    from_json(
      to_json(information),
      'STRUCT<
        api: STRUCT<
          version: INT,
          release: STRING,
          commit: STRING
        >,
        timestamp: STRING,
        tibia_urls: ARRAY<STRING>,
        status: STRUCT<
          error: INT,
          http_code: INT,
          message: STRING
        >
      >'
    ) AS information,
    cast(regexp_extract(_metadata.file_path, '([0-9]{4}-[0-9]{2}-[0-9]{2})', 1) AS DATE) AS source_file_date,
    _metadata.file_path AS source_file_path,
    _metadata.file_modification_time AS source_file_modified_at,
    current_timestamp() AS ingested_at
  FROM '/Volumes/tibia_analytics/bronze/raw/worlds'
)
FILEFORMAT = JSON
FORMAT_OPTIONS ('multiLine' = 'true')
COPY_OPTIONS ('mergeSchema' = 'false');